# The communication game

Companion notebook to Lecture 12, *Team games and hidden-role games*, parts 2 and 3 (correlation, imperfect recall, and the realization form). It checks the numbers the notes state for the communication game of Figure *The communication game*. All arithmetic is exact, with `Fraction` and SymPy. Every check is an `assert`, so running the notebook end to end confirms all of them.

The notebook reproduces worked examples and includes calculations used in the exercises. Try the exercises first if you want to solve them on your own.

**Requirements:** `sympy`.

In [1]:
import itertools
from fractions import Fraction

import sympy as sp

half = Fraction(1, 2)

## The game tree

The leaves are $z_1, \dots, z_{12}$, numbered as in the figure. Each leaf is a tuple `(bit, msg, guess2, guess3)`. Message `0` is the left message and guess `0` means guessing bit 0. If Player 2 guesses wrong, play stops at a 0 leaf and Player 3 does not move (`guess3 = None`). The team gets 1 if and only if Player 2 is right and Player 3 is wrong.

In [2]:
Z = [
    (0, 0, 0, 0), (0, 0, 0, 1), (0, 0, 1, None),   # z1  z2  z3
    (1, 0, 0, None), (1, 0, 1, 0), (1, 0, 1, 1),   # z4  z5  z6
    (0, 1, 0, 0), (0, 1, 0, 1), (0, 1, 1, None),   # z7  z8  z9
    (1, 1, 0, None), (1, 1, 1, 0), (1, 1, 1, 1),   # z10 z11 z12
]
u = [int(g3 is not None and g2 == bit and g3 != bit) for bit, msg, g2, g3 in Z]
p_chance = [half] * len(Z)
U_diag = [pc * uz for pc, uz in zip(p_chance, u)]   # diagonal of U = diag(p_chance(z) * u(z))

assert u == [0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0]   # leaf payoffs in the figure
for k, (z, uz) in enumerate(zip(Z, u), start=1):
    print(f"z{k:<2} (bit, msg, guess2, guess3) = {z}   u = {uz}")

z1  (bit, msg, guess2, guess3) = (0, 0, 0, 0)   u = 0
z2  (bit, msg, guess2, guess3) = (0, 0, 0, 1)   u = 1
z3  (bit, msg, guess2, guess3) = (0, 0, 1, None)   u = 0
z4  (bit, msg, guess2, guess3) = (1, 0, 0, None)   u = 0
z5  (bit, msg, guess2, guess3) = (1, 0, 1, 0)   u = 1
z6  (bit, msg, guess2, guess3) = (1, 0, 1, 1)   u = 0
z7  (bit, msg, guess2, guess3) = (0, 1, 0, 0)   u = 0
z8  (bit, msg, guess2, guess3) = (0, 1, 0, 1)   u = 1
z9  (bit, msg, guess2, guess3) = (0, 1, 1, None)   u = 0
z10 (bit, msg, guess2, guess3) = (1, 1, 0, None)   u = 0
z11 (bit, msg, guess2, guess3) = (1, 1, 1, 0)   u = 1
z12 (bit, msg, guess2, guess3) = (1, 1, 1, 1)   u = 0


A pure plan of Player 1 is a pair (message on bit 0, message on bit 1). A pure plan of Player 2 or Player 3 is a pair (guess after the left message, guess after the right message). `coords` and `plan` convert between pure plans and the coordinates $(a, b; c, d)$ of the notes. `play` evaluates a pure profile by walking the tree.

In [3]:
Pi1 = Pi2 = Pi3 = list(itertools.product((0, 1), repeat=2))
joint = list(itertools.product(Pi1, Pi2))           # Pi_1 x Pi_2
assert len(joint) == 16 and len(Pi3) == 4


def coords(p1, p2):
    # Joint plan -> (a, b; c, d). a = 1 iff the left message on bit 0, and so on.
    return (1 - p1[0], 1 - p1[1], 1 - p2[0], 1 - p2[1])


def plan(a, b, c, d):
    # (a, b; c, d) in {0, 1}^4 -> joint plan (pi_1, pi_2).
    return ((1 - a, 1 - b), (1 - c, 1 - d))


def play(p1, p2, p3):
    # Team's expected utility, computed directly from the tree.
    total = Fraction(0)
    for bit in (0, 1):
        msg = p1[bit]
        if p2[msg] == bit and p3[msg] != bit:
            total += half
    return total

## Realization vectors

Definition *Team realization vectors*. The entry $x^{\pi_1, \pi_2}[z]$ is 1 if and only if the team plays all of its actions on the path to $z$. For Player 3, $y[z]$ is 1 if and only if Player 3 plays its action on the path to $z$ (or does not act on it).

In [4]:
def x_vec(p1, p2):
    return tuple(int(p1[bit] == msg and p2[msg] == g2) for bit, msg, g2, g3 in Z)


def y_vec(p3):
    return tuple(1 if g3 is None else int(p3[msg] == g3) for bit, msg, g2, g3 in Z)


def xUy(x, y):
    return sum(w * xz * yz for w, xz, yz in zip(U_diag, x, y))


def show(v):
    # Print a vector of length 12 in the notes' layout, with bars between blocks of three.
    s = [str(Fraction(t)) for t in v]
    return "(" + " | ".join(", ".join(s[i:i + 3]) for i in range(0, 12, 3)) + ")"

Lemma *Utility in realization form* says that the expected utility equals $\boldsymbol{x}^\top U \boldsymbol{y}$. We check it against the direct tree evaluation for all $16 \cdot 4$ pure profiles.

In [5]:
for p1, p2 in joint:
    for p3 in Pi3:
        assert xUy(x_vec(p1, p2), y_vec(p3)) == play(p1, p2, p3)
print("x^T U y equals the tree evaluation for all", len(joint) * len(Pi3), "pure profiles")

x^T U y equals the tree evaluation for all 64 pure profiles


After Definition *Team realization vectors*, the notes say that the 16 joint plans give only 12 vectors. Two joint plans collide exactly when Player 1 always sends the same message, so Player 2's action after the other message is never used.

In [6]:
groups = {}
for p1, p2 in joint:
    groups.setdefault(x_vec(p1, p2), []).append((p1, p2))
V = set(groups)

collisions = [g for g in groups.values() if len(g) > 1]
assert len(V) == 12
assert len(collisions) == 4 and all(len(g) == 2 for g in collisions)
assert {jp for g in collisions for jp in g} == {(p1, p2) for p1, p2 in joint if p1[0] == p1[1]}
print("joint plans:", len(joint), "  distinct realization vectors:", len(V))
for g in collisions:
    print("  same vector for (a, b; c, d) =", " and ".join("(%d, %d; %d, %d)" % coords(*jp) for jp in g))

joint plans: 16   distinct realization vectors: 12
  same vector for (a, b; c, d) = (1, 1; 1, 1) and (1, 1; 1, 0)
  same vector for (a, b; c, d) = (1, 1; 0, 1) and (1, 1; 0, 0)
  same vector for (a, b; c, d) = (0, 0; 1, 1) and (0, 0; 0, 1)
  same vector for (a, b; c, d) = (0, 0; 1, 0) and (0, 0; 0, 0)


Example *Realization vectors in the communication game*. If both teammates always play left, the realization vector is $(1, 1, 0 \mid 1, 0, 0 \mid 0, 0, 0 \mid 0, 0, 0)$. We also compute $\boldsymbol{x}(a, b; c, d)$ and $\boldsymbol{y}$ from the tree and compare them with the formulas in the example. At pure coordinates they must agree with `x_vec` and `y_vec`.

In [7]:
a, b, c, d, e, f = sp.symbols("a b c d e f")
bar = lambda t: 1 - t
P1 = {(0, 0): a, (0, 1): bar(a), (1, 0): b, (1, 1): bar(b)}   # (bit, msg)
P2 = {(0, 0): c, (0, 1): bar(c), (1, 0): d, (1, 1): bar(d)}   # (msg, guess)
P3 = {(0, 0): e, (0, 1): bar(e), (1, 0): f, (1, 1): bar(f)}   # (msg, guess)

x_sym = [P1[bit, msg] * P2[msg, g2] for bit, msg, g2, g3 in Z]
y_sym = [sp.Integer(1) if g3 is None else P3[msg, g3] for bit, msg, g2, g3 in Z]

x_notes = [a*c, a*c, a*bar(c),  b*c, b*bar(c), b*bar(c),
           bar(a)*d, bar(a)*d, bar(a)*bar(d),  bar(b)*d, bar(b)*bar(d), bar(b)*bar(d)]
y_notes = [e, bar(e), 1,  1, e, bar(e),  f, bar(f), 1,  1, f, bar(f)]

assert x_vec(*plan(1, 1, 1, 1)) == (1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0)
assert all(sp.expand(s - t) == 0 for s, t in zip(x_sym, x_notes))
assert all(sp.expand(s - t) == 0 for s, t in zip(y_sym, y_notes))
for p1, p2 in joint:
    sub = dict(zip((a, b, c, d), coords(p1, p2)))
    assert tuple(int(t.subs(sub)) for t in x_sym) == x_vec(p1, p2)
for p3 in Pi3:
    sub = {e: 1 - p3[0], f: 1 - p3[1]}
    assert tuple(int(t.subs(sub)) for t in y_sym) == y_vec(p3)
print("all-left plan:", show(x_vec(*plan(1, 1, 1, 1))))
print("x(a, b; c, d) and y match the example")

all-left plan: (1, 1, 0 | 1, 0, 0 | 0, 0, 0 | 0, 0, 0)
x(a, b; c, d) and y match the example


## The one-time pad

Section *The one-time pad*. The pad is $\boldsymbol\mu^\star = \tfrac12 \delta_{(1, 0; 1, 0)} + \tfrac12 \delta_{(0, 1; 0, 1)}$. Under $\boldsymbol\mu^\star$, Player 2 always guesses right, and the message is uniform and independent of the bit.

In [8]:
pad = [plan(1, 0, 1, 0), plan(0, 1, 0, 1)]
mu_star = {pad[0]: half, pad[1]: half}

for p1, p2 in pad:
    assert all(p2[p1[bit]] == bit for bit in (0, 1))   # Player 2 is right on both bits
for bit, msg in itertools.product((0, 1), repeat=2):
    assert sum(w for (p1, _), w in mu_star.items() if p1[bit] == msg) == half
print("Player 2 always right, and P(message | bit) = 1/2 for every bit and message")

Player 2 always right, and P(message | bit) = 1/2 for every bit and message


The notes say the team gets $1/2$ against every $\boldsymbol{y}$. The payoff is linear in Player 3's mixed strategy, so it suffices to check Player 3's four pure plans.

In [9]:
def mu_value(mu, p3):
    return sum(w * play(p1, p2, p3) for (p1, p2), w in mu.items())


for p3 in Pi3:
    assert mu_value(mu_star, p3) == half
    print(f"Player 3 plan (e, f) = {(1 - p3[0], 1 - p3[1])}: team gets {mu_value(mu_star, p3)}")

Player 3 plan (e, f) = (1, 1): team gets 1/2
Player 3 plan (e, f) = (1, 0): team gets 1/2
Player 3 plan (e, f) = (0, 1): team gets 1/2
Player 3 plan (e, f) = (0, 0): team gets 1/2


Conversely, if Player 3 guesses uniformly ($e = f = 1/2$), no joint plan gets more than $1/2$, so no $\boldsymbol\mu$ does. Together with the pad this shows that the TMECor value is $1/2$, and that the maxmin and min-max values agree (Section *Correlation restores the minimax theorem*).

In [10]:
y_unif = tuple(sum(Fraction(y_vec(p3)[k]) for p3 in Pi3) / 4 for k in range(len(Z)))
assert list(y_unif) == [t.subs({e: half, f: half}) for t in y_sym]

best_vs_unif = max(xUy(x_vec(p1, p2), y_unif) for p1, p2 in joint)
assert best_vs_unif == half
tmecor_value = half   # lower bound from the pad, upper bound from y_unif
print("best joint plan against e = f = 1/2:", best_vs_unif)
print("TMECor value:", tmecor_value)

best joint plan against e = f = 1/2: 1/2
TMECor value: 1/2


## The TME value

Table *TME versus TMECor in the communication game*. The TME value is $1/4$ and is attained at $a = b = c = d = 1/2$. We check that this strategy guarantees exactly $1/4$. For the upper bound, let Player 3 copy Player 2's guess probabilities, $e = c$ and $f = d$. Then
$$\frac14 - u(a, b; c, d; c, d) = \frac{a + b}{2}\left(c - \frac12\right)^2 + \frac{2 - a - b}{2}\left(d - \frac12\right)^2,$$
and both coefficients are nonnegative since $a, b \in [0, 1]$. So no uncorrelated team strategy guarantees more than $1/4$. SymPy checks the identity.

In [11]:
def tme_objective(point):
    # min over Player 3's pure plans of x(a, b; c, d)^T U y. The min over mixed y is the same.
    av, bv, cv, dv = point
    x = [av*cv, av*cv, av*(1-cv),  bv*cv, bv*(1-cv), bv*(1-cv),
         (1-av)*dv, (1-av)*dv, (1-av)*(1-dv),  (1-bv)*dv, (1-bv)*(1-dv), (1-bv)*(1-dv)]
    return min(xUy(x, y_vec(p3)) for p3 in Pi3)


assert tme_objective((half, half, half, half)) == Fraction(1, 4)

# Upper bound: Player 3 copies Player 2, e = c and f = d.
u_sym = sp.expand(xUy(x_sym, y_sym))
u_copy = u_sym.subs({e: c, f: d})
certificate = (a + b) / 2 * (c - sp.Rational(1, 2))**2 + (2 - a - b) / 2 * (d - sp.Rational(1, 2))**2
assert sp.expand(sp.Rational(1, 4) - u_copy - certificate) == 0
tme_value = Fraction(1, 4)

assert tmecor_value == 2 * tme_value   # "twice the TME value"
print("a = b = c = d = 1/2 guarantees", tme_objective((half,) * 4))
print("1/4 - u(a, b; c, d; c, d) = (a + b)/2 (c - 1/2)^2 + (2 - a - b)/2 (d - 1/2)^2 >= 0")
print("TMECor / TME =", tmecor_value / tme_value)

a = b = c = d = 1/2 guarantees 1/4
1/4 - u(a, b; c, d; c, d) = (a + b)/2 (c - 1/2)^2 + (2 - a - b)/2 (d - 1/2)^2 >= 0
TMECor / TME = 2


## The pad is not uncorrelated

Example *Realization vectors in the communication game*. The realization vector of the pad is $\boldsymbol{x}^{\boldsymbol\mu^\star} = (\tfrac12, \tfrac12, 0 \mid 0, \tfrac12, \tfrac12 \mid \tfrac12, \tfrac12, 0 \mid 0, \tfrac12, \tfrac12)$, the midpoint of two points of $V$. No $(a, b; c, d)$ gives $\boldsymbol{x}(a, b; c, d) = \boldsymbol{x}^{\boldsymbol\mu^\star}$.

In [12]:
x_pad = tuple(sum(w * x_vec(p1, p2)[k] for (p1, p2), w in mu_star.items()) for k in range(len(Z)))
assert x_pad == tuple(Fraction(t) for t in (half, half, 0, 0, half, half, half, half, 0, 0, half, half))
assert all(x_vec(*jp) in V for jp in pad)
assert sp.solve([s - sp.Rational(t) for s, t in zip(x_sym, x_pad)], [a, b, c, d], dict=True) == []
print("x^mu* =", show(x_pad))
print("no (a, b; c, d) realizes it")

x^mu* = (1/2, 1/2, 0 | 0, 1/2, 1/2 | 1/2, 1/2, 0 | 0, 1/2, 1/2)
no (a, b; c, d) realizes it


## The flow relaxation

Example *The flow relaxation is too large*. The polytope $R$ is cut out by $\boldsymbol{x} \ge 0$, by $x[z_1] + x[z_3] + x[z_7] + x[z_9] = 1$ and $x[z_4] + x[z_5] + x[z_{10}] + x[z_{11}] = 1$, and by $x[z_1] = x[z_2]$, $x[z_5] = x[z_6]$, $x[z_7] = x[z_8]$, $x[z_{11}] = x[z_{12}]$. The four inequalities of the table are $x[z_1] + x[z_5] \le 1$, $x[z_3] + x[z_4] \le 1$, $x[z_7] + x[z_{11}] \le 1$ and $x[z_9] + x[z_{10}] \le 1$. Below, `z(k)` is the 0-based index of $z_k$.

In [13]:
n = len(Z)
z = lambda k: k - 1


def row(*ks, coef=None):
    r = [Fraction(0)] * n
    for k, cf in zip(ks, coef or [1] * len(ks)):
        r[z(k)] += cf
    return r


A_eq = [row(1, 3, 7, 9), row(4, 5, 10, 11),
        row(1, 2, coef=[1, -1]), row(5, 6, coef=[1, -1]),
        row(7, 8, coef=[1, -1]), row(11, 12, coef=[1, -1])]
b_eq = [1, 1, 0, 0, 0, 0]

nonneg = [(row(k, coef=[-1]), 0) for k in range(1, n + 1)]           # -x[z] <= 0
table = [((1, 5), row(1, 5)), ((3, 4), row(3, 4)), ((7, 11), row(7, 11)), ((9, 10), row(9, 10))]
cuts = [(r, 1) for _, r in table]                                     # x[z_i] + x[z_j] <= 1

We enumerate vertices exactly. A vertex is a feasible point where the equalities and some tight inequalities have a unique solution.

In [14]:
def solve_unique(rows, rhs):
    # Exact Gaussian elimination. Returns the unique solution, or None.
    M = [list(r) + [Fraction(v)] for r, v in zip(rows, rhs)]
    piv_row = 0
    for col in range(n):
        p = next((i for i in range(piv_row, len(M)) if M[i][col] != 0), None)
        if p is None:
            return None
        M[piv_row], M[p] = M[p], M[piv_row]
        M[piv_row] = [t / M[piv_row][col] for t in M[piv_row]]
        for i in range(len(M)):
            if i != piv_row and M[i][col] != 0:
                M[i] = [s - M[i][col] * t for s, t in zip(M[i], M[piv_row])]
        piv_row += 1
    if any(r[-1] != 0 for r in M[n:]):
        return None
    return tuple(M[i][-1] for i in range(n))


def satisfies(x, ineqs):
    return all(sum(r_ * x_ for r_, x_ in zip(r, x)) <= bound for r, bound in ineqs)


def vertices(ineqs):
    k = n - len(A_eq)   # the six equalities are independent
    found = set()
    for S in itertools.combinations(ineqs, k):
        x = solve_unique(A_eq + [r for r, _ in S], b_eq + [bound for _, bound in S])
        if x is not None and satisfies(x, ineqs):
            found.add(x)
    return found


R_vertices = vertices(nonneg)
print("vertices of R:", len(R_vertices))

vertices of R:

 16


The notes say that $R$ has 16 vertices, one for each pair of a bit-0 outcome and a bit-1 outcome. Here an outcome is a child of a Player 2 node. It is either a 0 leaf or a Player 3 node with its two leaves.

In [15]:
twin = {1: 2, 5: 6, 7: 8, 11: 12}
outcomes0, outcomes1 = (1, 3, 7, 9), (4, 5, 10, 11)


def pair_vertex(i, j):
    leaves = {i, j, twin.get(i), twin.get(j)} - {None}
    return tuple(Fraction(int(k in leaves)) for k in range(1, n + 1))


pairs = {pair_vertex(i, j): (i, j) for i in outcomes0 for j in outcomes1}
assert len(R_vertices) == 16 and set(pairs) == R_vertices
print("the 16 vertices of R are the 16 outcome pairs")

the 16 vertices of R are the 16 outcome pairs

A joint plan produces every pair except the four in the table. Each inequality of the table cuts off its own vertex and no point of $V$.

In [16]:
V_frac = {tuple(Fraction(t) for t in v) for v in V}
extra = {pairs[v] for v in R_vertices - V_frac}
assert V_frac <= R_vertices
assert extra == {ij for ij, _ in table}

for ij, r in table:
    violated = {pairs[v] for v in R_vertices if not satisfies(v, [(r, 1)])}
    assert violated == {ij}
    print(f"x[z{ij[0]}] + x[z{ij[1]}] <= 1 cuts off only the vertex (z{ij[0]}, z{ij[1]})")
assert all(satisfies(v, cuts) for v in V_frac)

x[z1] + x[z5] <= 1 cuts off only the vertex (z1, z5)
x[z3] + x[z4] <= 1 cuts off only the vertex (z3, z4)
x[z7] + x[z11] <= 1 cuts off only the vertex (z7, z11)
x[z9] + x[z10] <= 1 cuts off only the vertex (z9, z10)


Adding the four inequalities to $R$ gives exactly $\mathcal X$, so the vertex set becomes exactly $V$.

In [17]:
X_vertices = vertices(nonneg + cuts)
assert X_vertices == V_frac
print("vertices of R with the four inequalities:", len(X_vertices), "= |V|")

vertices of R with the four inequalities: 12 = |V|


After the example, the notes say that the maxmin value over $R$ is also $1/2$. Against $e = f = 1/2$, every vertex of $R$ gets at most $1/2$, hence so does every point of $R$. The pad lies in $\mathcal X \subseteq R$ and gets $1/2$ against every $\boldsymbol{y}$.

In [18]:
assert max(xUy(v, y_unif) for v in R_vertices) == half
assert satisfies(x_pad, nonneg) and all(sum(r_ * x_ for r_, x_ in zip(r, x_pad)) == bv
                                        for r, bv in zip(A_eq, b_eq))
assert all(xUy(x_pad, y_vec(p3)) == half for p3 in Pi3)
print("maxmin value over R:", half)

maxmin value over R: 1/2


## A further problem

The notes leave out this problem to keep the lecture short. It describes the uncorrelated set $\widetilde{\mathcal X}$ exactly. Let $R$ be the flow relaxation of Example *The flow relaxation is too large*, and write $x_i := x[z_i]$. Show that
$$\widetilde{\mathcal X} = \{\boldsymbol{x} \in R : x_1 x_5 = x_3 x_4, \quad x_7 x_{11} = x_9 x_{10}\}.$$
Interpret each equation as a statement about Player 2's conditional behavior, and use the result to show again that the pad is not in $\widetilde{\mathcal X}$.

*Hint.* By Section *Convexifying the TME*, $\widetilde{\mathcal X}$ is the set of vectors $\boldsymbol{x}(a, b; c, d)$. For the reverse inclusion, set $a := x_1 + x_3$ and $b := x_4 + x_5$, and treat the cases $a = 0$ and $b = 0$ separately.

The cell below checks the easy direction symbolically, and that the pad breaks the first equation.

In [ ]:
q1 = lambda v: v[0] * v[4] - v[2] * v[3]     # x1 x5 - x3 x4
q2 = lambda v: v[6] * v[10] - v[8] * v[9]    # x7 x11 - x9 x10
assert sp.expand(q1(x_sym)) == 0 and sp.expand(q2(x_sym)) == 0
assert q1(x_pad) == Fraction(1, 4) and q2(x_pad) == Fraction(1, 4)
print("x(a, b; c, d) satisfies both equations, and the pad gives", q1(x_pad), "and", q2(x_pad))